# Web-Based RAG System
## Retrieval-Augmented Generation from Web Pages

This notebook demonstrates a complete RAG system that:
1. Fetches content from URLs
2. Chunks text with overlap
3. Creates embeddings and FAISS index
4. Retrieves relevant chunks
5. Generates answers using LLM

## Setup: Import Libraries and Load API Key

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if API key is set
if os.environ.get('GROQ_API_KEY'):
    print("✅ GROQ_API_KEY is set")
else:
    print("❌ GROQ_API_KEY is not set")
    print("Please create a .env file with your GROQ_API_KEY")

In [ ]:
import requests
import re
from bs4 import BeautifulSoup
from typing import List, Dict, Tuple

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss
import numpy as np

print("✅ All libraries imported successfully")

## Step 1: Configure URLs to Index

In [ ]:
# Define URLs to fetch
# You can change these to any publicly accessible URLs
urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Machine_learning",
]

print(f"URLs to index: {len(urls)}")
for i, url in enumerate(urls, 1):
    print(f"  {i}. {url}")

## Step 2: Initialize Models

In [ ]:
# Initialize embeddings model
print("🔧 Initializing embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embeddings model ready")

# Initialize LLM
print("\n🔧 Initializing LLM...")
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
print("✅ LLM ready")

## Step 3: Fetch Content from URLs

In [ ]:
def fetch_url(url: str) -> str:
    """
    Fetch and extract main text from a URL
    """
    try:
        print(f"📥 Fetching: {url}")
        response = requests.get(url, timeout=10, headers={
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        response.raise_for_status()
        
        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove script, style, nav, header, footer elements
        for element in soup(['script', 'style', 'nav', 'header', 'footer', 
                             'aside', 'iframe', 'noscript']):
            element.decompose()
        
        # Extract text from main content areas
        main_content = soup.find('main') or soup.find('article') or \
                      soup.find('div', {'id': re.compile(r'content|main', re.I)}) or \
                      soup.find('div', {'class': re.compile(r'content|main', re.I)}) or \
                      soup.body
        
        if main_content:
            text = main_content.get_text(separator=' ', strip=True)
        else:
            text = soup.get_text(separator=' ', strip=True)
        
        # Clean up whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        print(f"✅ Extracted {len(text):,} characters")
        return text
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return ""

# Fetch all URLs
print("\n" + "="*60)
print("FETCHING WEB PAGES")
print("="*60 + "\n")

url_content = {}
for url in urls:
    content = fetch_url(url)
    if content:
        url_content[url] = content

print(f"\n✅ Successfully fetched {len(url_content)}/{len(urls)} URLs")

## Step 4: Chunk the Documents

In [ ]:
print("\n" + "="*60)
print("CHUNKING TEXT")
print("="*60 + "\n")

# Initialize text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

chunks = []
chunk_sources = []  # Track source URL for each chunk

for url, content in url_content.items():
    # Create document
    doc = Document(page_content=content, metadata={"source": url})
    
    # Split into chunks
    doc_chunks = splitter.split_documents([doc])
    
    print(f"📄 {url}")
    print(f"   └─ Created {len(doc_chunks)} chunks")
    
    chunks.extend(doc_chunks)
    chunk_sources.extend([url] * len(doc_chunks))

print(f"\n✅ Total chunks created: {len(chunks)}")

## Step 5: Create Embeddings and Build FAISS Index

In [ ]:
print("\n" + "="*60)
print("BUILDING VECTOR INDEX")
print("="*60 + "\n")

# Extract text from chunks
texts = [chunk.page_content for chunk in chunks]

# Create embeddings
print("🔄 Creating embeddings...")
embeddings_list = embeddings.embed_documents(texts)

# Convert to numpy array
embeddings_array = np.array(embeddings_list, dtype='float32')
print(f"✅ Created {len(embeddings_list)} embeddings")

# Create FAISS index
print("\n🔄 Building FAISS index...")
dimension = 384  # Dimension for all-MiniLM-L6-v2
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings_array)

print(f"✅ FAISS index built with {faiss_index.ntotal} vectors")

## Step 6: Define Query Function

In [ ]:
def ask_question(query: str, k: int = 3):
    """
    Ask a question and get an answer based on retrieved context
    """
    print("\n" + "="*60)
    print(f"QUERY: {query}")
    print("="*60 + "\n")
    
    # Embed query
    query_embedding = np.array([embeddings.embed_query(query)], dtype='float32')
    
    # Search FAISS index
    print(f"🔍 Retrieving top-{k} relevant chunks...\n")
    distances, indices = faiss_index.search(query_embedding, k)
    
    # Display evidence
    print("📚 EVIDENCE:\n")
    retrieved_chunks = []
    
    for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
        chunk_text = chunks[idx].page_content
        source_url = chunk_sources[idx]
        
        print(f"[{i}] Source: {source_url}")
        print(f"    Similarity Score: {distance:.4f}")
        print(f"    Text: {chunk_text[:200]}...")
        print()
        
        retrieved_chunks.append((chunk_text, source_url))
    
    # Build context for LLM
    context_parts = []
    for chunk_text, source_url in retrieved_chunks:
        context_parts.append(f"[Source: {source_url}]\n{chunk_text}")
    
    context = "\n\n".join(context_parts)
    
    # Generate answer
    print("🤖 Generating answer...\n")
    prompt = f"""Answer the following question based on the provided context. 
If the answer cannot be found in the context, say so.

Context:
{context}

Question: {query}

Answer:"""
    
    response = llm.invoke(prompt)
    answer = response.content
    
    print("="*60)
    print("💡 ANSWER:")
    print("="*60)
    print(answer)
    print("\n" + "="*60 + "\n")
    
    return answer

## Step 7: Test with Sample Questions

In [ ]:
# Example question 1
ask_question("What is artificial intelligence?")

In [ ]:
# Example question 2
ask_question("What is the difference between machine learning and deep learning?")

In [ ]:
# Example question 3
ask_question("Who are the pioneers of AI?")

## Step 8: Try Your Own Questions

In [ ]:
# Ask your own question here
your_question = "What are some applications of machine learning?"
ask_question(your_question, k=5)  # Retrieve top 5 chunks

## Summary

This notebook demonstrated a complete Web-based RAG system:

1. ✅ Fetched content from multiple URLs
2. ✅ Extracted main text (removed scripts, navigation, etc.)
3. ✅ Chunked text with sliding window and overlap
4. ✅ Created embeddings using HuggingFace
5. ✅ Built FAISS vector index
6. ✅ Retrieved top-k relevant chunks
7. ✅ Displayed evidence (chunk text + source URL)
8. ✅ Generated answers using Groq LLM with context

### Next Steps

- Try different URLs (course pages, documentation, Wikipedia articles)
- Experiment with chunk_size and chunk_overlap
- Adjust k to retrieve more or fewer chunks
- Add more URLs to expand your knowledge base